# Construction du dataset SFT

Objectif : construire le dataset SFT final à partir de MediQAl, FrenchMedMCQA et MedQuAD (voir `notebooks/01_exploration_sources.ipynb`), en trois étapes : nettoyage et dédoublonnage, répartition en train/validation/test/eval_clinique, puis sous-échantillonnage à environ 5000 paires.

Ce notebook exécute et documente les fonctions de `scripts/extraction.py` (`clean_sft_dataset`, `assign_splits`, `subsample_sft_dataset`), pour garder une trace des transformations appliquées (auditabilité RGPD, voir `docs/etape1.md`).

In [1]:
import sys
sys.path.append("..")

from collections import Counter
from dotenv import load_dotenv
from scripts.extraction import (
    load_mediqa,
    load_frenchmedmcqa,
    load_medquad,
    clean_sft_dataset,
    build_sft_dataset,
    subsample_sft_dataset,
    build_sft_sample,
    TAILLE_CIBLE_SFT,
)
load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Nettoyage et dédoublonnage

Doublons exacts (instruction, reponse), source par source. L'exploration qui a mené à cette logique est dans `notebooks/01_exploration_sources.ipynb` : 48 doublons exacts repérés dans MedQuAD, 1 doublon exact dans FrenchMedMCQA, aucun dans MediQAl.

### MedQuAD : 48 doublons exacts

48 lignes partagent exactement le même couple (question, réponse) que la ligne `instruction`/`reponse` construite dans `load_medquad`.

In [2]:
medquad = load_medquad()
print("medquad brut :", len(medquad))

cles = [(r["instruction"], r["reponse"]) for r in medquad]
compteur = Counter(cles)
doublons = {cle: n for cle, n in compteur.items() if n > 1}
print("couples (instruction, reponse) en double :", len(doublons))
print("lignes en trop à retirer :", sum(n - 1 for n in doublons.values()))

medquad brut : 16407
couples (instruction, reponse) en double : 32
lignes en trop à retirer : 48


Trois exemples de doublons, pour voir à quoi ils ressemblent :

In [3]:
for instruction, reponse in list(doublons.keys())[:3]:
    print("instruction :", instruction)
    print("reponse     :", reponse[:150].replace("\n", " "), "...")
    print()

instruction : What is (are) Hypoglycemia ?
reponse     : Hypoglycemia, also called low blood glucose or low blood sugar, occurs when blood glucose drops below normal levels. Glucose, an important source of e ...

instruction : What are the symptoms of Hypoglycemia ?
reponse     : Hypoglycemia causes symptoms such as                  - hunger  - shakiness  - nervousness  - sweating  - dizziness or light-headedness  - sleepiness  ...

instruction : What causes Hypoglycemia ?
reponse     : Diabetes Medications                  Hypoglycemia can occur as a side effect of some diabetes medications, including insulin and oral diabetes medica ...



Comme repéré dans le notebook 01, ce sont des questions génériques qui reviennent (« What is (are) Hypoglycemia ? » posée deux fois avec la même réponse), pas des variantes à garder : ce sont bien des doublons exacts à retirer.

### FrenchMedMCQA : 1 doublon exact

Un seul doublon, déjà repéré dans le notebook d'exploration : une question sur Streptococcus pneumoniae apparaît deux fois dans le split train, avec le même id.

In [4]:
frenchmedmcqa = load_frenchmedmcqa()
print("frenchmedmcqa brut :", len(frenchmedmcqa))

cles = [(r["instruction"], r["reponse"]) for r in frenchmedmcqa]
compteur = Counter(cles)
doublons = {cle: n for cle, n in compteur.items() if n > 1}
print("couples en double :", len(doublons))

for instruction, reponse in doublons.keys():
    print("instruction :", instruction)
    print("reponse     :", reponse)

frenchmedmcqa brut : 1080
couples en double : 1
instruction : Parmi les caractères suivants, lequel n'est pas retrouvé chez Streptococcus pneumoniae ?
a) Présence d'une capsule
b) Catalase négative
c) Survie en aérobiose et anaérobiose
d) Production d'une toxine de Panton-Valentine
e) Hémolytique sur gélose au sang
reponse     : d) Production d'une toxine de Panton-Valentine


### MediQAl : aucun doublon

Confirmation sur l'agrégat construit par `load_mediqa`, cohérente avec le notebook d'exploration (qui incluait déjà le cas clinique dans la comparaison).

In [5]:
mediqa = load_mediqa()
print("mediqa brut :", len(mediqa))

cles = [(r["instruction"], r["reponse"]) for r in mediqa]
print("couples uniques :", len(set(cles)))

mediqa brut : 4969
couples uniques : 4969


### Agrégat nettoyé

`build_sft_dataset` charge les trois sources, applique `clean_sft_dataset` (retire les doublons exacts en gardant la première occurrence), puis répartit en splits avec `assign_splits` (section suivante).

In [6]:
agregat_brut = mediqa + frenchmedmcqa + medquad
agregat = build_sft_dataset()

print("agrégat brut    :", len(agregat_brut))
print("agrégat nettoyé :", len(agregat))
print("doublons retirés :", len(agregat_brut) - len(agregat))

retires_par_source = Counter()
vus = set()
for r in agregat_brut:
    cle = (r["instruction"], r["reponse"])
    if cle in vus:
        retires_par_source[r["source"]] += 1
    else:
        vus.add(cle)
print("répartition des doublons retirés :", dict(retires_par_source))

agrégat brut    : 22456
agrégat nettoyé : 22407
doublons retirés : 49
répartition des doublons retirés : {'frenchmedmcqa': 1, 'medquad': 48}


22456 exemples SFT bruts, 49 doublons exacts retirés (48 dans MedQuAD, 1 dans FrenchMedMCQA), 22407 exemples dans l'agrégat nettoyé. Aucun doublon cross-source détecté, ce qui est attendu vu que les langues et les sujets diffèrent d'une source à l'autre.

## Répartition en train / validation / test / eval_clinique

`assign_splits` répartit l'agrégat nettoyé en quatre jeux, stratifié par source (mediqal, frenchmedmcqa, medquad) : chaque source suit les mêmes proportions de split, pour éviter qu'un split se retrouve dominé par une seule source ou une seule langue. Le split déjà présent sur FrenchMedMCQA (hérité du dataset source) est écrasé pour que toutes les sources suivent la même logique. Les proportions retenues : 80% train, 10% validation, 5% test, 5% eval_clinique (choix documenté dans `docs/decisions.md`).

`agregat` (variable ci-dessus) est déjà réparti en splits, puisque `build_sft_dataset` enchaîne nettoyage et répartition.

In [7]:
repartition = Counter(r["split"] for r in agregat)
for split, n in repartition.items():
    print(f"{split:15s} {n:6d}  ({n / len(agregat):.1%})")

train            17925  (80.0%)
validation        2241  (10.0%)
eval_clinique     1121  (5.0%)
test              1120  (5.0%)


In [8]:
repartition_par_source = Counter((r["source"], r["split"]) for r in agregat)
sources = sorted(set(r["source"] for r in agregat))
splits = ["train", "validation", "test", "eval_clinique"]

for source in sources:
    total_source = sum(n for (s, _), n in repartition_par_source.items() if s == source)
    print(source, "-", total_source, "exemples")
    for split in splits:
        n = repartition_par_source[(source, split)]
        print(f"  {split:15s} {n:6d}  ({n / total_source:.1%})")

frenchmedmcqa - 1079 exemples
  train              863  (80.0%)
  validation         108  (10.0%)
  test                54  (5.0%)
  eval_clinique       54  (5.0%)
mediqal - 4969 exemples
  train             3975  (80.0%)
  validation         497  (10.0%)
  test               248  (5.0%)
  eval_clinique      249  (5.0%)
medquad - 16359 exemples
  train            13087  (80.0%)
  validation        1636  (10.0%)
  test               818  (5.0%)
  eval_clinique      818  (5.0%)


Les proportions par source suivent bien les ratios visés (80/10/5/5), aux arrondis près sur les petites sources.

### Vérification : pas de fuite entre splits

Comme la répartition se fait par simple partition des indices après mélange, il ne peut pas y avoir de recouvrement d'IDs entre splits. On le vérifie quand même explicitement, pour garder une preuve dans ce notebook.

In [9]:
ids_par_split = {split: set(r["id"] for r in agregat if r["split"] == split) for split in splits}

for i, split_a in enumerate(splits):
    for split_b in splits[i + 1:]:
        intersection = ids_par_split[split_a] & ids_par_split[split_b]
        print(f"{split_a} / {split_b} : {len(intersection)} id en commun")

train / validation : 0 id en commun
train / test : 0 id en commun
train / eval_clinique : 0 id en commun
validation / test : 0 id en commun
validation / eval_clinique : 0 id en commun
test / eval_clinique : 0 id en commun


L'agrégat SFT est réparti en train, validation, test et eval_clinique, stratifié par source, sans aucune fuite d'ID entre splits.

## Sous-échantillonnage à ~5000 paires

`subsample_sft_dataset` réduit l'agrégat réparti à environ `TAILLE_CIBLE_SFT` paires (cible fixée par la mission, voir `docs/etape1.md`). Le tirage se fait par fraction proportionnelle dans chaque (source, split) : chaque stratum perd la même proportion d'exemples, ce qui préserve à la fois les ratios de splits (80/10/5/5) et les proportions actuelles des sources (donc des langues). Ce choix est documenté dans `docs/decisions.md`, avec un point ouvert à trancher avec le mentor : cette approche garde les proportions naturelles des sources, ce qui donne un dataset à dominante anglaise (MedQuAD), pas un 50/50 fr/en.

In [10]:
echantillon = subsample_sft_dataset(agregat)
print("taille cible :", TAILLE_CIBLE_SFT)
print("taille obtenue :", len(echantillon))

taille cible : 5000
taille obtenue : 5001


### Vérification : proportions préservées

Comparaison des proportions par source et par split, avant et après sous-échantillonnage.

In [11]:
repartition_apres = Counter((r["source"], r["split"]) for r in echantillon)

for source in sources:
    total_avant = sum(n for (s, _), n in repartition_par_source.items() if s == source)
    total_apres = sum(n for (s, _), n in repartition_apres.items() if s == source)
    print(f"{source:15s} avant {total_avant / len(agregat):.1%}   après {total_apres / len(echantillon):.1%}")

print()
for split in splits:
    total_avant = sum(n for (_, sp), n in repartition_par_source.items() if sp == split)
    total_apres = sum(n for (_, sp), n in repartition_apres.items() if sp == split)
    print(f"{split:15s} avant {total_avant / len(agregat):.1%}   après {total_apres / len(echantillon):.1%}")

frenchmedmcqa   avant 4.8%   après 4.8%
mediqal         avant 22.2%   après 22.2%
medquad         avant 73.0%   après 73.0%

train           avant 80.0%   après 80.0%
validation      avant 10.0%   après 10.0%
test            avant 5.0%   après 5.0%
eval_clinique   avant 5.0%   après 5.0%


Les proportions par source et par split restent stables aux arrondis près : le sous-échantillonnage ne déséquilibre ni les langues ni les splits par rapport à l'agrégat de départ.

### Vérification : pas de doublon introduit

Le tirage se fait sans remise à l'intérieur de chaque stratum, donc aucun exemple ne peut apparaître deux fois.

In [12]:
ids = [r["id"] for r in echantillon]
print("exemples :", len(ids))
print("ids uniques :", len(set(ids)))

exemples : 5001
ids uniques : 5001


### Détail par (source, split)

In [13]:
for source in sources:
    print(source)
    for split in splits:
        avant = repartition_par_source[(source, split)]
        apres = repartition_apres[(source, split)]
        print(f"  {split:15s} {avant:6d} -> {apres:5d}")

frenchmedmcqa
  train              863 ->   193
  validation         108 ->    24
  test                54 ->    12
  eval_clinique       54 ->    12
mediqal
  train             3975 ->   887
  validation         497 ->   111
  test               248 ->    55
  eval_clinique      249 ->    56
medquad
  train            13087 ->  2920
  validation        1636 ->   365
  test               818 ->   183
  eval_clinique      818 ->   183


## Synthèse

Pipeline SFT complet (`build_sft_sample`, qui enchaîne `build_sft_dataset` et `subsample_sft_dataset`) : 22456 exemples bruts, 49 doublons exacts retirés (22407 exemples nettoyés), répartis en train/validation/test/eval_clinique stratifié par source sans fuite d'ID, puis sous-échantillonnés à environ 5000 paires en préservant les proportions de sources et de splits.

Point à valider avec le mentor (voir `docs/decisions.md`) : garder cette proportion naturelle (environ 73% anglais / 27% français) ou plafonner MedQuAD pour se rapprocher d'un équilibre 50/50 entre français et anglais.

Prochaine étape : anonymiser SFT et DPO avec Presidio.

In [14]:
verification = build_sft_sample()
print("build_sft_sample() :", len(verification), "exemples")

build_sft_sample() : 5001 exemples
